# **NLP Practical 03**

Write functions for predicting a word / sentence :

1. 1-gram
2. 2-gram
3. 3-gram
4. Using Laplace add on
5. Witten Bell

In [ ]:
import nltk
from collections import Counter
from nltk.util import ngrams
from nltk.lm import WittenBellInterpolated
from nltk.lm.preprocessing import padded_everygram_pipeline

nltk.download('punkt')
nltk.download('punkt_tab')

In [ ]:
# Training Corpus
text = """
machine learning is a branch of artificial intelligence
machine learning uses data to learn patterns
machine learning models can predict future results
deep learning is a part of machine learning
artificial intelligence helps computers solve problems
data science uses machine learning for prediction
machine learning is useful in many applications
"""

In [ ]:
# Tokenization
words = nltk.word_tokenize(text.lower())

print("=" * 60)
print("TRAINING CORPUS")
print("=" * 60)

print(" ".join(words))

In [ ]:
# Create N-Grams

unigrams = list(ngrams(words, 1))
bigrams = list(ngrams(words, 2))
trigrams = list(ngrams(words, 3))


unigram_counts = Counter(unigrams)
bigram_counts = Counter(bigrams)
trigram_counts = Counter(trigrams)

vocabulary = set(words)

print("\nVocabulary size:", len(vocabulary))


# 1-GRAM MODEL
def predict_1gram():
    """
    Predict the most frequent word in the corpus.
    """

    most_common = unigram_counts.most_common(1)

    return most_common[0][0][0]



# 2-GRAM MODEL
def predict_2gram(previous_word):
    """
    Predict the next word using the previous word.
    """

    candidates = {}

    for (w1, w2), count in bigram_counts.items():

        if w1 == previous_word.lower():

            candidates[w2] = count

    if not candidates:
        return "No prediction available"

    return max(candidates, key=candidates.get)



# 3-GRAM MODEL
def predict_3gram(previous_two_words):
    """
    Predict the next word using the previous two words.
    """

    w1, w2 = previous_two_words.lower().split()

    candidates = {}

    for (a, b, c), count in trigram_counts.items():

        if a == w1 and b == w2:

            candidates[c] = count

    if not candidates:
        return "No prediction available"

    return max(candidates, key=candidates.get)

In [ ]:
# LAPALCE ADD-ONE MODEL

def laplace_probability(previous_word, next_word):
    """
    Calculate bigram probability using Laplace Add-One smoothing.

    P(next_word | previous_word)
    =
    (Count(previous_word, next_word) + 1)
    /
    (Count(previous_word) + Vocabulary Size)
    """

    previous_word = previous_word.lower()
    next_word = next_word.lower()

    bigram_count = bigram_counts[(previous_word, next_word)]

    previous_count = unigram_counts[(previous_word,)]

    V = len(vocabulary)

    probability = (bigram_count + 1) / (previous_count + V)

    return probability


def predict_laplace(previous_word):
    """
    Predict next word using Laplace Add-One smoothing.
    """

    previous_word = previous_word.lower()

    best_word = None
    best_probability = 0

    for word in vocabulary:

        probability = laplace_probability(
            previous_word,
            word
        )

        if probability > best_probability:

            best_probability = probability
            best_word = word

    return best_word, best_probability


# WITTEN-BELL MODEL
# Use padded corpus for NLTK Witten-Bell model

train_data, padded_sents = padded_everygram_pipeline(
    2,
    [words]
)

witten_bell_model = WittenBellInterpolated(2)

witten_bell_model.fit(train_data, vocabulary_text=words)


def predict_witten_bell(previous_word):
    """
    Predict next word using Witten-Bell smoothing.
    """

    previous_word = previous_word.lower()

    best_word = None
    best_probability = 0

    for word in vocabulary:

        probability = witten_bell_model.score(
            word,
            [previous_word]
        )

        if probability > best_probability:

            best_probability = probability
            best_word = word

    return best_word, best_probability

In [ ]:
# Predict a sentence
def generate_sentence(model_type, start_words, length=8):

    sentence = start_words.lower().split()

    for i in range(length):

        if model_type == "1gram":

            next_word = predict_1gram()

        elif model_type == "2gram":

            next_word = predict_2gram(sentence[-1])

            if next_word == "No prediction available":
                break

        elif model_type == "3gram":

            if len(sentence) < 2:
                break

            next_word = predict_3gram(
                sentence[-2] + " " + sentence[-1]
            )

            if next_word == "No prediction available":
                break

        elif model_type == "laplace":

            next_word, probability = predict_laplace(
                sentence[-1]
            )

        elif model_type == "witten-bell":

            next_word, probability = predict_witten_bell(
                sentence[-1]
            )

        sentence.append(next_word)

    return " ".join(sentence)


# Display results

print("\n")
print("=" * 60)
print("WORD PREDICTION RESULTS")
print("=" * 60)


# 1-Gram

print("\n1-GRAM")
print("-" * 60)

print("Prediction:", predict_1gram())


# 2-Gram

print("\n2-GRAM")
print("-" * 60)

previous_word = "machine"

prediction = predict_2gram(previous_word)

print("Given word :", previous_word)
print("Predicted next word:", prediction)


# 3-Gram

print("\n3-GRAM")
print("-" * 60)

previous_two_words = "machine learning"

prediction = predict_3gram(previous_two_words)

print("Given words :", previous_two_words)
print("Predicted next word:", prediction)


# Laplace

print("\nLAPLACE ADD-ONE")
print("-" * 60)

previous_word = "machine"

prediction, probability = predict_laplace(previous_word)

print("Given word :", previous_word)
print("Predicted next word:", prediction)
print("Probability :", round(probability, 4))


# Witten-Bell

print("\nWITTEN-BELL")
print("-" * 60)

previous_word = "machine"

prediction, probability = predict_witten_bell(previous_word)

print("Given word :", previous_word)
print("Predicted next word:", prediction)
print("Probability :", round(probability, 4))